In [ ]:
import cohere
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm

api_key = 'COHERE_API_KEY'
co = cohere.Client(api_key)

In [59]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

texts = text.split('.')
texts = [t.strip() for t in texts]
print(texts, '\n', len(texts))

['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan', 'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine', 'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind', 'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007', 'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar', 'Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm', 'Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles', 'Interstellar uses extensive practical and mini

In [60]:
# 문장 임베딩
response = co.embed(
  texts=texts,
  input_type="search_document",
  model="embed-v4.0",
).embeddings

embeds = np.array(response)
print(embeds.shape) # 1536차원의 15개의 벡터
print(embeds.dtype)

(15, 1536)
float64


In [61]:
# 검색 인덱스 구축
# FAISS IndexFlatL2를 임베딩 차원인 1536차원 기준으로 초기화한 다음, 여기에 임베딩된 벡터들을 저장. 단, FAISS는 원문 텍스트를 저장하지 않고 벡터만 저장하므로, 검색 결과 인덱스로 원문 texts를 따로 매핑
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim) #IndexFlatL2 : 유클리드 거리 기반으로 최근접 이웃을 찾는 가장 기본적인 인덱스 생성
index.add(np.float32(embeds)) # 벡터룰 추가하는 add, 벡터를 검색하는 search 메서드 둘다 float32를 기대함

In [62]:
# 밀집 검색
def search(query, number_of_result=3):
    #쿼리 임베딩 생성
    query_embed = co.embed(texts=[query], input_type="search_query", model='embed-v4.0',).embeddings[0]
    print(len(query_embed))
    # 최근접 이웃을 추출
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_result)

    # Dataframe을 사용하여 출력 준비
    texts_np = np.array(texts) # 인덱싱을 쉽게 하기 위해 텍스트 리스트 → 넘파이 배열
    results = pd.DataFrame(data={'텍스트': texts_np[similar_item_ids[0]], '거리': distances[0]})

    # 결과 출력 및 리턴
    print(f"Query:'{query}'\n최근접 이웃:")
    return results


In [63]:
query = 'How precise was the science'
results = search(query)
results

1536
Query:'How precise was the science'
최근접 이웃:


,텍스트,거리
0,It has also received praise from many astronom...,1.298189
1,Caltech theoretical physicist and 2017 Nobel l...,1.565775
2,"Since its premiere, Interstellar gained a cult...",1.630452


In [64]:
# 어휘 검색
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words # "the", "a", "an", "is", "are", "and", "or", "to" 와 같은 불용어들
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation) # 구두점들 strip
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

def keyword_search(query, top_k=3, num_candidate=15):
    print("Query: ",query)

    # BM25 검색(어휘 검색)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidate)[-num_candidate:] # NumPy에서 상위 k개 / 하위 k개 인덱스를 빠르게 찾을 때 쓰는 함수, 기본적으로 오름차순, 정렬하지 않고 서칭만
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x:x['score'], reverse=True)

    print(f"Top-3 어휘 검색(BM25) 결과")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace('\n',' ')))

keyword_search("how precise was the science")

100%|██████████| 15/15 [00:00<?, ?it/s]

Query:  how precise was the science
Top-3 어휘 검색(BM25) 결과
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


In [65]:
query = 'how precise was the science'
results = co.rerank(query=query, documents=texts, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15239799),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time'), index=13, relevance_score=0.03255596),


In [66]:
# 키워드 검색 + 리랭킹
def keyword_and_reranking_search(query, top_k=3, num_candidate=10):
    print("Query: ", query)

    # BM25 검색 
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidate)[-num_candidate:]
    bm25_hits = [{'corpus_id':idx, 'score':bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print("Top-3 어휘 검색 (BM25) 결과")
    for hit in bm25_hits[:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    # 리랭킹 추가
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\n리랭킹으로 얻은 top-3 결과 ({len(bm25_hits)}개의 BM25 결과를 재조정)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace('\n', " ")))
keyword_and_reranking_search(query='how precise was the science')

Query:  how precise was the science
Top-3 어휘 검색 (BM25) 결과
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles

리랭킹으로 얻은 top-3 결과 (10개의 BM25 결과를 재조정)
	0.152	It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
	0.050	The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


In [67]:
# LLM API을 사용한 근거 기반 생성
query = "income generated"
# 1. 검색, 여기서는 임베딩 검색을 사용하지만, 하이브리드 방식이 이상적
results = search(query)

# 2. 근거 기반 생성
docs_dict=[{'text':text} for text in results['텍스트']]
response = co.chat(
    message = query,
    documents = docs_dict
)
print(response.text)
response

1536
Query:'income generated'
최근접 이웃:
The film Interstellar generated a worldwide gross of over $677 million, or $773 million including subsequent re-releases, making it the tenth-highest grossing film of 2014.


NonStreamedChatResponse(text='The film Interstellar generated a worldwide gross of over $677 million, or $773 million including subsequent re-releases, making it the tenth-highest grossing film of 2014.', generation_id='701050f1-b60b-47b6-a913-8278a1decece', response_id='848c5991-cd9f-4bb8-9177-8d67707c7d7a', citations=[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'), ChatCitation(start=34, end=70, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=75, end=120, text='$773 million including subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=136, end=172, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_2', 'text': 'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'}, {'id': 'doc_0', 'text': 'The film had a worldwide gross ov

In [68]:
response.citations

[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'),
 ChatCitation(start=34, end=70, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=75, end=120, text='$773 million including subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=136, end=172, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')]

In [69]:
# 로컬 모델을 사용한 RAG
from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model_path = hf_hub_download(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="Phi-3-mini-4k-instruct-q4.gguf"
)

llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=4096,
    seed=42,
    verbose=False,
)


embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)

db = FAISS.from_texts(texts, embedding_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12437.29it/s]


In [70]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)
rag.invoke("Income generated")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Income generated',
 'result': ' The information provided does not directly address the income generated by "Interstellar." However, considering its acclaim and production quality as described (extensive practical effects, high-end cinematography), it can be inferred that "Interstellar" likely performed well financially. Yet, without specific financial data or box office figures, we cannot determine the exact income generated from this information alone.'}

In [71]:
query = "Income generated"

retriever = db.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar

--- Retrieved Chunk 2 ---
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

--- Retrieved Chunk 3 ---
In the United States, it was first released on film stock, expanding to venues using digital projectors

--- Retrieved Chunk 4 ---
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm

--- Retrieved Chunk 5 ---
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014


In [72]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


retriever = db.as_retriever(search_kwargs={'k':10})
cross_encoder = HuggingFaceCrossEncoder(
    model_name='BAAI/bge-reranker-base'
)
compressor = CrossEncoderReranker( #리랭커
    model = cross_encoder,
    top_n=3
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor =compressor,
    base_retriever=retriever
)

template = """<|user|>
Relevant information:
{context}

Provide a concise answer to the following question using only the relevant information provided above.
If the answer is not in the relevant information, say you don't know.

Question:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,  
    chain_type_kwargs={
        "prompt": prompt
    },
    return_source_documents=True,
    verbose=True
)

result = rag.invoke({"query": "Income generated"})

print(result["result"])

for i, doc in enumerate(result["source_documents"]):
    print(f"\n--- Source {i+1} ---")
    print(doc.page_content)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9136.63it/s]




> Entering new RetrievalQA chain...

> Finished chain.
 The income generated by Interstellar is over $677 million worldwide, and with subsequent re-releases, it totaled approximately $773 million.

--- Source 1 ---
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan

--- Source 2 ---
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007

--- Source 3 ---
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014


In [73]:
from sentence_transformers import CrossEncoder
from langchain_core.prompts import PromptTemplate

reranker = CrossEncoder("BAAI/bge-reranker-base")

template = """<|user|>
Relevant information:
{context}

Provide a concise answer to the following question using only the relevant information provided above.
If the answer is not in the relevant information, say you don't know.

Question:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

def rag_with_reranker(query, retrieve_k=10, rerank_top_n=3):
    docs = db.as_retriever(
        search_kwargs={"k": retrieve_k}
    ).invoke(query) # FAISS에 저장된 벡터 중 쿼리와 관련성이 높은 10개의 문서 추출

    print("\n[Before rerank]")
    for i, doc in enumerate(docs):
        print(f"\n--- Retrieved {i+1} ---")
        print(doc.page_content)

    # 리랭커 입력: query-document pair
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)

    # 리랭킹
    reranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    print("\n[After rerank]")
    for i, (doc, score) in enumerate(reranked[:rerank_top_n]):
        print(f"\n--- Reranked {i+1} | score: {score:.4f} ---")
        print(doc.page_content)

    # 상위 청크 3개만 context로 사용
    top_docs = [doc for doc, score in reranked[:rerank_top_n]]
    context = "\n\n".join(doc.page_content for doc in top_docs)

    # 최종 답변 생성
    final_prompt = prompt.format(
        context=context,
        question=query
    )

    answer = llm.invoke(final_prompt)

    return {
        "query": query,
        "result": answer,
        "source_documents": top_docs
    }

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8738.41it/s]


In [74]:
result = rag_with_reranker("Income generated")

print("\nAnswer:")
print(result["result"])


[Before rerank]

--- Retrieved 1 ---
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar

--- Retrieved 2 ---
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

--- Retrieved 3 ---
In the United States, it was first released on film stock, expanding to venues using digital projectors

--- Retrieved 4 ---
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm

--- Retrieved 5 ---
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014

--- Retrieved 6 ---
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight

--- Retrieved 7 ---
It has also recei